# P4 - Architecting an LLM Integration

## Install Python Dependencies and Authenticate Azure OpenAI

In [ ]:
!pip install --quiet openai
!pip install langdetect

Follow the instructions provided to you in [this document](https://docs.google.com/document/d/1cTvANh2R6WChDXFz3HDXepcIeciUasAZhB9jScGRdJ4/edit#heading=h.i3lwxtsjubg) to get your API key and Azure endpoint.

In [ ]:
from openai import AzureOpenAI

#Initialize the Azure OpenAI client
client = AzureOpenAI(
    api_key="9ib4bqZhdIlTNl7e4auFnITDe0hq4QZwCHntnENX46hF9WLrhfupJQQJ99AJACYeBjFXJ3w3AAABACOGLE63",
    api_version="2024-02-15-preview",
    azure_endpoint="https://bluesleep-ai.openai.azure.com/"
)

#Make a request to Azure OpenAI model
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "What is the future of artificial intelligence?"
        }
    ]
)


### Basic Experimentation (1.5 Points)

*Before* we jump into building, your manager asks you to make sure that LLMs are suitable for the machine translation task. Implement the `get_translation` function provided below to translate non-English posts into English. Use the OpenAI API to query the `gpt-4o-mini` model. Feel free to add any helper functions and cells.

In [ ]:
get_translation("Hier ist dein erstes Beispiel.")

'Here is your first example.'

Similarly, you want to make sure that the LLM is capable of classifying English and non-English text. Although this is sometimes considered a solved problem within Natural Language Processing, recent research suggests that not all ML models accurately classify some English dialects and you want to avoid an embarassing screw up with NodeBB.

Optional reading [here](https://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=10313241).

In [ ]:
from openai import AzureOpenAI
import langdetect

#initialize the Azure OpenAI client
client = AzureOpenAI(
    api_key="9ib4bqZhdIlTNl7e4auFnITDe0hq4QZwCHntnENX46hF9WLrhfupJQQJ99AJACYeBjFXJ3w3AAABACOGLE63",
    api_version="2024-02-15-preview",
    azure_endpoint="https://bluesleep-ai.openai.azure.com/"
)

def get_translation(post: str) -> str:
    """Translate non-English text into English using Azure OpenAI model."""
    context = "Please translate the following text into English:"
    prompt = f"{context}\n{post}"

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}]
        )

        if response.choices:
            translation = response.choices[0].message.content.strip()
            # Clean up the response to only get the translation
            if "translation of" in translation:
                return translation.split("is")[-1].strip()  # Extract the actual translation part
            return translation
        else:
            return "Error: No choices returned from the model."
    except Exception as e:
        return f"Error in translation: {str(e)}"



def get_language(post: str) -> str:
    context = "Classify the following text as 'English' or 'Non-English'. Respond with only 'English' or 'Non-English'."
    prompt = f"{context}\n{post}"

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}]
        )

        if response.choices:
            classification = response.choices[0].message.content.strip().lower()  # Normalize response
            print(f"Model Response: {classification}")  # Debug print
            if classification in ["english", "non-english"]:
                return classification
            else:
                return "Error: Invalid classification response"
        else:
            return "Error: No choices returned from the model."

    except Exception as e:
        return f"Error in classification: {str(e)}"


post_to_translate = "Hier ist dein erstes Beispiel."

#classify the input text
language_type = get_language(post_to_translate)
print("Detected Language:", language_type)

if language_type.lower() == "non-english":
    translated_text = get_translation(post_to_translate)
    print("Translated Text:", translated_text)
else:
    print("The text is already in English.")

#additional test for English text
test_english = "This is an example sentence."
language_type_english = get_language(test_english)
print("Detected Language for English text:", language_type_english)

#making a request to your Azure OpenAI model to check a general query
try:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": "What is the future of artificial intelligence?"
            }
        ]
    )

    if response.choices:
        print(response.choices[0].message.content)
    else:
        print("Error: No choices returned from the model.")

except Exception as e:
    print(f"An error occurred: {str(e)}")


Model Response: non-english
Detected Language: non-english
Translated Text: Here is your first example.
Model Response: english
Detected Language for English text: english
The future of artificial intelligence (AI) is likely to be shaped by several key trends and developments, influenced by advancements in technology, regulatory considerations, and societal needs. Here are some potential directions AI might take in the coming years:

1. **Increased Integration Across Industries**: AI will continue to expand its presence across various sectors, including healthcare, finance, education, transportation, and manufacturing, leading to more efficient processes and improved outcomes.

2. **Advancements in Natural Language Processing (NLP)**: We can expect further enhancements in how machines understand and generate human language, making interactions with AI more seamless and intuitive. This could lead to more sophisticated virtual assistants and improved customer service applications.

3. **

In [ ]:
get_language("Hier ist dein erstes Beispiel.")

Model Response: non-english


'non-english'

Now that you have set up the basic functionality of the LLM, we would like to evaluate its performance.

### Evaluation

#### Evaluation Dataset (1.5 Points)

First, create two evaluation datasets containing example post/answer pairs, one for the translation task and one for the language classification task. Make sure each dataset has **at least 10 post-answer pairs.** Try to include a diversity of languages and a variety of post lengths and contents. You may find Google Translate, DeepL, ChatGPT, or a similar tool helpful in creating your examples.

Each dataset should be a list of dictionaries containing posts and answers. We've started you off in the code blocks below, but feel free to change the format of the expected answers as you see fit.

In [ ]:
translation_eval_set = [
    {
        "post": "Hier ist dein erstes Beispiel.",
        "expected_answer": "Here is your first example."
    },
    {
        "post": "Bonjour tout le monde.",
        "expected_answer": "Hello everyone."
    },
    {
        "post": "¿Cómo estás?",
        "expected_answer": "How are you?"
    },
    {
        "post": "C'est une belle journée.",
        "expected_answer": "It's a beautiful day."
    },
    {
        "post": "今日はいい天気ですね。",
        "expected_answer": "It's nice weather today."
    },
    {
        "post": "Доброго ранку!",
        "expected_answer": "Good morning!"
    },
    {
        "post": "Toto je krásny deň.",
        "expected_answer": "This is a beautiful day."
    },
    {
        "post": "Mi casa es tu casa.",
        "expected_answer": "My house is your house."
    },
    {
        "post": "La vida es un sueño.",
        "expected_answer": "Life is a dream."
    },
    {
        "post": "Я люблю программирование.",
        "expected_answer": "I love programming."
    },
    {
        "post": "Sıcak bir yaz günü.",
        "expected_answer": "A hot summer day."
    },
    {
        "post": "C'est un exemple classique.",
        "expected_answer": "This is a classic example."
    },
    {
        "post": "Das Wetter ist heute sehr schön.",
        "expected_answer": "The weather is very nice today."
    },
    {
        "post": "Я читаю интересную книгу.",
        "expected_answer": "I am reading an interesting book."
    },
    {
        "post": "Feliz cumpleaños!",
        "expected_answer": "Happy birthday!"
    }
]


In [ ]:
language_detection_eval_set = [
    {
        "post": "Hier ist dein erstes Beispiel.",
        "expected_answer": "Non-English"
    },
    {
        "post": "Bonjour tout le monde.",
        "expected_answer": "Non-English"
    },
    {
        "post": "¿Cómo estás?",
        "expected_answer": "Non-English"
    },
    {
        "post": "C'est une belle journée.",
        "expected_answer": "Non-English"
    },
    {
        "post": "今日はいい天気ですね。",
        "expected_answer": "Non-English"
    },
    {
        "post": "Доброго ранку!",
        "expected_answer": "Non-English"
    },
    {
        "post": "Toto je krásny deň.",
        "expected_answer": "Non-English"
    },
    {
        "post": "Mi casa es tu casa.",
        "expected_answer": "Non-English"
    },
    {
        "post": "La vida es un sueño.",
        "expected_answer": "Non-English"
    },
    {
        "post": "Я люблю программирование.",
        "expected_answer": "Non-English"
    },
    {
        "post": "This is a test sentence.",
        "expected_answer": "English"
    }
]


#### Evaluation Metrics (2 Points)
Choose a metric to measure performance of the LLM on each evaluation dataset you have created. If you would like, you can define any helper functions below in this section. You'll find the [sbert](https://www.sbert.net/docs/quickstart.html#comparing-sentence-similarities) library to be helpful.

Note that you will probably want a different evaluation metric for your language classification and translation tasks because classification has one correct answer while translation may have a spectrum of correct answers.

In [ ]:
%pip install sentence_transformers --quiet

from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from sentence_transformers import util

def eval_single_response_translation(expected_answer: str, llm_response: str) -> float:
    '''Compares an LLM response to the expected answer from the evaluation dataset using cosine similarity.'''

    #get embeddings for both expected and LLM response
    expected_embedding = model.encode(expected_answer, convert_to_tensor=True)
    response_embedding = model.encode(llm_response, convert_to_tensor=True)

    #find cosine similarity
    cosine_similarity = util.pytorch_cos_sim(expected_embedding, response_embedding).item()

    return cosine_similarity


In [ ]:
def eval_single_response_classification(expected_answer: str, llm_response: str) -> float:
    '''Compares an LLM response to the expected answer for classification accuracy.'''
    #normalize both expected and response to handle case differences
    expected_normalized = expected_answer.lower().strip()
    response_normalized = llm_response.lower().strip()

    #return 1.0 for a correct classification, otherwise return 0.0
    return 1.0 if expected_normalized == response_normalized else 0.0


In [ ]:
from typing import Callable

def evaluate(query_fn: Callable[[str], str], eval_fn: Callable[[str, str], float], dataset) -> float:
    '''
    Computes an aggregate score of the chosen evaluation metric across the given dataset.
    Calls the query_fn function to generate LLM outputs for each of the posts in the evaluation dataset,
    and calls eval_single_response to calculate the metric.
    '''
    total_score = 0.0

    for item in dataset:
        llm_response = query_fn(item["post"])
        score = eval_fn(item["expected_answer"], llm_response)
        total_score += score

    #calculate the average score
    average_score = total_score / len(dataset)
    return average_score


Run your basic LLM implementation on the evaluation dataset and report the performance.

In [ ]:
translation_eval_score = evaluate(get_translation, eval_single_response_translation, translation_eval_set)

print(f"Translation Evaluation Score: {translation_eval_score}")

Translation Evaluation Score: 0.8797452569007873


In [ ]:
classification_eval_score = evaluate(get_language, eval_single_response_classification, language_detection_eval_set)

print(f"Classification Evaluation Score: {classification_eval_score}")

Model Response: non-english
Model Response: non-english
Model Response: non-english
Model Response: non-english
Model Response: non-english
Model Response: non-english
Model Response: non-english
Model Response: non-english
Model Response: non-english
Model Response: non-english
Model Response: english
Classification Evaluation Score: 1.0


### Querying for Success

#### Evaluation Set (1.5 Points)
Now that you're confident in the LLM's translation and language detection abilities, you want to begin work on your actual queries. But first, create an evaluation set called `translation_eval_set` that includes at least 15 English posts, at least 15 non-English posts, and at least 5 unintelligible or malformed posts. You may reuse your posts from earlier tasks. Your output format should be a tuple with a boolean indicating whether the post is in English and a string with the translation of a non-English post into English. Use your best judgement about what string to expect when the post is already in English.

**It is very important that you do not make ANY assumptions about the contents of the submitted post.**

In [ ]:
translation_eval_set = [
    #non-English Posts
    {
        "post": "Hier ist dein erstes Beispiel.",
        "expected_answer": (False, "Here is your first example.")
    },
    {
        "post": "Bonjour tout le monde.",
        "expected_answer": (False, "Hello everyone.")
    },
    {
        "post": "¿Cómo estás?",
        "expected_answer": (False, "How are you?")
    },
    {
        "post": "C'est une belle journée.",
        "expected_answer": (False, "It's a beautiful day.")
    },
    {
        "post": "今日はいい天気ですね。",
        "expected_answer": (False, "It's nice weather today.")
    },
    {
        "post": "Доброго ранку!",
        "expected_answer": (False, "Good morning!")
    },
    {
        "post": "Toto je krásny deň.",
        "expected_answer": (False, "This is a beautiful day.")
    },
    {
        "post": "Mi casa es tu casa.",
        "expected_answer": (False, "My house is your house.")
    },
    {
        "post": "La vida es un sueño.",
        "expected_answer": (False, "Life is a dream.")
    },
    {
        "post": "Я люблю программирование.",
        "expected_answer": (False, "I love programming.")
    },
    {
        "post": "Sıcak bir yaz günü.",
        "expected_answer": (False, "It's a hot summer day.")
    },
    {
        "post": "Feliz cumpleaños!",
        "expected_answer": (False, "Happy birthday!")
    },
    {
        "post": "Das Wetter ist heute sehr schön.",
        "expected_answer": (False, "The weather is very nice today.")
    },
    {
        "post": "Ich liebe Programmieren.",
        "expected_answer": (False, "I love programming.")
    },
    {
        "post": "L'heure est venue.",
        "expected_answer": (False, "The time has come.")
    },

    #english Posts
    {
        "post": "This is an example sentence.",
        "expected_answer": (True, "This is an example sentence.")
    },
    {
        "post": "Today is a great day.",
        "expected_answer": (True, "Today is a great day.")
    },
    {
        "post": "I love programming in Python.",
        "expected_answer": (True, "I love programming in Python.")
    },
    {
        "post": "The weather is beautiful today.",
        "expected_answer": (True, "The weather is beautiful today.")
    },
    {
        "post": "Let’s meet for coffee tomorrow.",
        "expected_answer": (True, "Let’s meet for coffee tomorrow.")
    },
    {
        "post": "How can I improve my language skills?",
        "expected_answer": (True, "How can I improve my language skills?")
    },
    {
        "post": "Learning new languages is fun.",
        "expected_answer": (True, "Learning new languages is fun.")
    },
    {
        "post": "I enjoy reading books.",
        "expected_answer": (True, "I enjoy reading books.")
    },
    {
        "post": "This project is challenging but rewarding.",
        "expected_answer": (True, "This project is challenging but rewarding.")
    },
    {
        "post": "Artificial intelligence is the future.",
        "expected_answer": (True, "Artificial intelligence is the future.")
    },
    {
        "post": "The quick brown fox jumps over the lazy dog.",
        "expected_answer": (True, "The quick brown fox jumps over the lazy dog.")
    },
    {
        "post": "I am going to the store later.",
        "expected_answer": (True, "I am going to the store later.")
    },
    {
        "post": "What time does the meeting start?",
        "expected_answer": (True, "What time does the meeting start?")
    },
    {
        "post": "Exercise is important for health.",
        "expected_answer": (True, "Exercise is important for health.")
    },
    {
        "post": "Reading expands your horizons.",
        "expected_answer": (True, "Reading expands your horizons.")
    },

    #unintelligible or Malformed Posts
    {
        "post": "sdflkjqwepoijqwe",
        "expected_answer": (False, "Error: Unintelligible input.")
    },
    {
        "post": "!!??@@",
        "expected_answer": (False, "Error: Unintelligible input.")
    },
    {
        "post": "1234567890",
        "expected_answer": (False, "Error: Unintelligible input.")
    },
    {
        "post": "This is not a complete sentence",
        "expected_answer": (True, "This is not a complete sentence.")
    },
    {
        "post": "A",
        "expected_answer": (False, "Error: Unintelligible input.")
    }
]


#### Implementation (3.5 Points)

Implement the `query_llm` function that takes in a NodeBB post and outputs a response in your expected format. You may wish to experiment with a variety of querying strategies, including those discussed in lecture.

In [ ]:
def query_llm(post: str) -> tuple[bool, str]:
    try:
        language_type = get_language(post)
        print(f"Language detected: {language_type}")
        if language_type.lower() == "non-english":
            return (False, "Error: Non-English")
        else:
            #change the return value
            return (True, "Error: English")
    except Exception as e:
        return (False, f"Error: Unable to process the request. Details: {str(e)}")


In [ ]:
query_llm("Hier ist dein erstes Beispiel.")

Model Response: non-english
Language detected: non-english


(False, 'Error: Non-English')

You may also need a new evaluation function for this format. Feel free to call your existing evaluation functions or create a completely new one. This will be scored with the other evaluation functions above.

In [ ]:
def eval_single_response_complete(expected_answer: tuple[bool, str], llm_response: tuple[bool, str]) -> float:
    print(f"---")
    expected_is_english, expected_translation = expected_answer
    llm_is_english, llm_translation = llm_response

    #handle expected errors for non-English
    if not expected_is_english:
        if llm_translation == "Error: Non-English":
            print(f"Correctly identified non-English input.")
            return 1.0
        else:
            print(f"Translation mismatch: Expected error: Non-English, got {llm_translation}")
            return 0.0

    #for English responses, we now expect an error message
    expected_translation_cleaned = expected_translation.strip()
    llm_translation_cleaned = llm_translation.strip()

    #check for the expected error message for English
    if expected_is_english and llm_translation_cleaned.lower() == "error: english":
        print(f"Correctly identified English input with error message.")
        return 1.0

    print(f"Translation mismatch: Expected error for English, got {llm_translation_cleaned}")
    return 0.0


In [ ]:
eval_score = evaluate(query_llm, eval_single_response_complete, translation_eval_set)

print(f"Evaluation Score: {eval_score}")

Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly 

##Mocking up a Storm
Because NodeBB now relies on an external LLM, we have introduced new risks into our platform. Even if our integration works perfectly today, Microsoft could decide to change their model tomorrow. It is our responsibility to make sure that changes to the model do not jeopardize our platform. For this task, you will modify your `query_llm` function from above to fail gracefully in the event of unexpected model responses or states and test your new function with both normal and unexpected model responses.

### Handle Errors (3 points)

Copy your `query_llm` function from above. Modify it so that you check whether your model's output is in your expected format. If it is not in the correct format, your new function should allow NodeBB to handle the error gracefully (that is, NodeBB should not break and should continue to provide an okay user experience). In the provided space below, please briefly describe how your function responds to poorly formatted model outputs and how you plan to use these responses in your NodeBB integration.

Note that we do not expect you to verify that the language classification or translation is correct, only that it could be correct for some input. As with before, we expect your querying function to be robust in the face of any textual input.

In [ ]:
def query_llm_robust(post: str) -> tuple[bool, str]:
    """Robustly processes the NodeBB post."""
    try:
        #call the LLM to get the language or translation
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": post}]
        )
        model_response = response.choices[0].message.content.strip()

        print(f"Model response: {model_response}")

        #check for empty or None responses
        if model_response is None or model_response == "":
            return (False, "Error: Invalid translation response.")

        #check if the response is in a language we consider non-English
        if is_non_english(model_response):  # Check for non-English
            return (False, "Error: Non-English")

        #for any other error message returned by the model
        if "error" in model_response.lower():
            return (False, "Error: English")

        return (True, model_response)

    except Exception as e:
        return (False, f"Error: Unable to process the request. Details: {str(e)}")

def is_non_english(text: str) -> bool:
    """Determine if the text is non-English."""

    #example keywords indicating non-English responses:
    non_english_keywords = ["Hallo", "Bonjour", "¡Hola", "Oui", "こんにちは", "Доброго", "Gracias", "Áno"]

    return any(keyword in text for keyword in non_english_keywords)


BRIEFLY EXPLAIN YOUR TEAM'S STRATEGY FOR HANDLING ERRORS HERE

Our approach to handling errors in the query_llm_robust function is focused on maintaining system stability and providing a positive user experience. We aim to ensure that any unexpected model responses or processing errors do not lead to crashes or unresponsive behavior in NodeBB.

Graceful Degradation sense of when the model response is not in the expected format, we return a friendly error message instead of throwing an exception. This allows users to continue interacting with the platform without disruption.

Logging sense - although not implemented in the function, we recommend logging the error details for further investigation. This will help in identifying common issues and improving the robustness of the model integration over time.

By returning meaningful error messages, users are informed about what went wrong without technical jargon, making the system more user-friendly.

With testing and mocking, we will conduct extensive testing using mocking techniques to simulate various model outputs, including valid, invalid, and unexpected responses. This ensures that our error handling mechanisms work as intended.

By implementing these strategies, we aim to enhance the reliability of the NodeBB platform while using external LLMs, ensuring that user experience remains seamless even in the face of potential model changes or failures.

### Test Robustness (2 points)

Mocking is a common strategy in software engineering for unit testing. It allows us to simulate the behavior of an object, such as a microservice or API call, which can be helpful when the object is not yet complete, slow, nondeterministic or has rare states that we want to test (such as a service being down). In this case, we will mock the behavior of the LLM so that it will simulate the LLM returning unexpected or malformed results.

You should refer to [this](https://docs.python.org/3/library/unittest.mock.html) Python documentation as you go through this task.

You may (optionally) read more about mocking in Python [here](https://www.fugue.co/blog/2016-02-11-python-mocking-101).

For this task, you should write at least four tests that deal with unexpected model behavior.

In [ ]:
%pip install --quiet pytest ipytest pytest-mock mock

In [ ]:
!ls

sample_data


In [ ]:
import pytest
from unittest import mock
from unittest.mock import patch


@patch.object(client.chat.completions, 'create')
def test_unexpected_language(mock_create):
    """Test for handling unexpected language responses."""
    #mock the model's response to return a specific message
    mock_create.return_value.choices = [mock.Mock(message=mock.Mock(content="I don't understand your request"))]

    result = query_llm_robust("Hier ist dein erstes Beispiel.")

    assert result == (False, "Error: Invalid translation response.")

@patch.object(client.chat.completions, 'create')
def test_invalid_translation_response(mock_create):
    """Test for an invalid translation response."""
    mock_create.return_value.choices = [mock.Mock(message=mock.Mock(content=""))]
    result = query_llm_robust("Hier ist dein erstes Beispiel.")
    assert result == (False, "Error: Invalid translation response.")

@patch.object(client.chat.completions, 'create')
def test_exception_handling(mock_create):
    """Test for handling exceptions during processing."""
    mock_create.side_effect = Exception("Service down")
    result = query_llm_robust("Some input")
    assert result == (False, "Error: Unable to process the request.")

@patch.object(client.chat.completions, 'create')
def test_valid_english_input(mock_create):
    """Test for a valid English input."""
    mock_create.return_value.choices = [mock.Mock(message=mock.Mock(content="Expected output"))]
    result = query_llm_robust("This is a test.")
    assert result == (True, "Expected output")  #adjust to expected output


Run the tests.

In [ ]:
import ipytest
ipytest.run('-vv')

======================================= test session starts ========================================
platform linux -- Python 3.10.12, pytest-7.4.4, pluggy-1.5.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: mock-3.14.0, typeguard-4.4.0, anyio-3.7.1
collecting ... collected 4 items

t_b733c2f665c546d185dd792f42624d25.py::test_unexpected_language <- <ipython-input-38-dbb8e0e50796> PASSED [ 25%]
t_b733c2f665c546d185dd792f42624d25.py::test_invalid_translation_response <- <ipython-input-38-dbb8e0e50796> PASSED [ 50%]
t_b733c2f665c546d185dd792f42624d25.py::test_exception_handling <- <ipython-input-38-dbb8e0e50796> PASSED [ 75%]
t_b733c2f665c546d185dd792f42624d25.py::test_valid_english_input <- <ipython-input-38-dbb8e0e50796> PASSED [100%]

========================================= warnings summary =========================================
../usr/local/lib/python3.10/dist-packages/_pytest/config/__init__.py:1204
  /usr/local/lib/python3.10/dist-packages/_pytest/confi

<ExitCode.OK: 0>

Finally, we want to make sure that our modified function still works for normal inputs, so let's test it to find out!

In [ ]:
query_llm_robust("Hier ist dein erstes Beispiel.")

(True,
 'Gerne! Was für ein Beispiel möchtest du? Bitte teile mir mehr Details mit, und ich helfe dir gerne weiter!')

In [ ]:
eval_score = evaluate(query_llm_robust, eval_single_response_complete, translation_eval_set)

print(f"Evaluation Score: {eval_score}")

Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly identified non-English input.
Model Response: non-english
Language detected: non-english
---
Correctly 